https://mermaid.ai/d/58908990-2096-4e72-997d-6a769d04407b 



In [2]:
!pip install pymupdf

In [3]:
%pip install psycopg[binary]

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, sys

print("Working directory:", os.getcwd())
print("\nFiles here:", os.listdir("."))
print("\nSys path:", sys.path)

Working directory: d:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project

Files here: ['.gitignore', 'chunking.py', 'config.py', 'db.py', 'Deep+Learning+Ian+Goodfellow.pdf', 'demo.ipynb', 'embeddings.py', 'fullsystem.ipynb', 'pdf_loader.py', 'rag_api.py', 'requirements.txt', 'rerank.py', 'retrieval.py', 'sql_queries.py', 'system.png', 'task.ipynb', '__pycache__']

Sys path: ['c:\\Users\\Administrator\\anaconda3\\python313.zip', 'c:\\Users\\Administrator\\anaconda3\\DLLs', 'c:\\Users\\Administrator\\anaconda3\\Lib', 'c:\\Users\\Administrator\\anaconda3', '', 'c:\\Users\\Administrator\\anaconda3\\Lib\\site-packages', 'c:\\Users\\Administrator\\anaconda3\\Lib\\site-packages\\win32', 'c:\\Users\\Administrator\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\Administrator\\anaconda3\\Lib\\site-packages\\Pythonwin']


In [5]:
# install pgvector dependency required by db module
%pip install pgvector

In [24]:
import numpy as np

from config import *
from pdf_loader import read_pdf_text
from chunking import split_sections,chunk_text
from embeddings import load_embedder, embed_chunks
from db import *
from retrieval import retrieve_topk
from rerank import rerank_results
from chunking import build_chunks

In [25]:
from db import delete_all_chunks,count_chunks,preview_chunks
count_chunks(PG_CONN_STR)
preview_chunks(PG_CONN_STR)

📦 Total stored chunks: 3480

🔎 Stored Chunks Preview

DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 1 ]
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 2 ] deep learning ian goodfellow yoshua bengio aaron courville
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 3 ] contents website vii acknowledgments viii notation xi 1 introduction 1 1. 1 who should read this book?.................... 8 1. 2 historical trends in deep learning................. 11 i ap
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
.................... 36 2. 4 linear dependence and span.................... 37 2. 5 norms................................. 39 2. 6 special kinds of matrices and vectors............... 40 2. 7 eigendec
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
singular value decomposition......

In [ ]:
delete_all_chunks(PG_CONN_STR)

🗑️ All chunks deleted from rag_cv_chunks


In [12]:
count_chunks(PG_CONN_STR)
preview_chunks(PG_CONN_STR)

📦 Total stored chunks: 0

🔎 Stored Chunks Preview



In [6]:
print("🔎 Loading embedding model...")
embedder = load_embedder(EMBED_MODEL_NAME)

dim = embedder.get_sentence_embedding_dimension()

from db import init_db

# =========================================
# 2️ INIT DATABASE
# =========================================

print("🧱 Initializing database...")
init_db(PG_CONN_STR, dim)


🔎 Loading embedding model...
🧱 Initializing database...


init_db (done)>> get data (done)>> split to chunks  
split to chunks 
1) split text or pdf to (sections or (chatpers and subscetion) ) يعتمد على فهمك للداتا الي معك 
2) split sections to chunks 
3) chunks content to embedder 
4) embeddeing to database
userQ >> matching with DB >> retrive top K >>> send to llm >> gen.final answer 

In [7]:
# =========================================
# 3️ INGEST CV
# =========================================

pdf_path = r"Deep+Learning+Ian+Goodfellow.pdf"

doc_name = "DeepLearning-IanGoodfellow_RAG"

print("📄 Reading PDF...")
text = read_pdf_text(pdf_path)

📄 Reading PDF...


In [8]:
# =========================================
# SAVE TEXT FILE
# =========================================

txt_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text saved to:", txt_path)

✅ Text saved to: D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book.txt


In [10]:
!pip install pymupdf

In [9]:
import fitz

print(fitz.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.13 running on win32 (64-bit).



In [11]:
import fitz  # PyMuPDF binding: PDF/ebook/document handling (open, read, extract text)
import re

def read_pdf_text2(pdf_path: str):
    """
    Read a PDF and return cleaned text.
    fitz is used to open and parse pages for text extraction.
    """

    # Open PDF document
    doc = fitz.open(pdf_path)

    pages_text = []

    # Iterate pages and extract text
    for i, page in enumerate(doc):
        print(f"Reading page {i+1}/{len(doc)}")

        txt = page.get_text()  # extract page text

        # clean null / control chars and normalize whitespace
        txt = txt.replace("\x00", " ")
        txt = re.sub(r"[ \t]+", " ", txt)

        # keep page boundary markers
        pages_text.append(f"\n[PAGE {i+1}]\n{txt}")

    # join all pages
    text = "\n\n".join(pages_text)

    # collapse many empty lines to at most 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [12]:

pdf_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep+Learning+Ian+Goodfellow.pdf"
doc_name = "DeepLearning-IanGoodfellow_RAG"


In [13]:
# =========================================
# 3️ INGEST CV
# =========================================

print("📄 Reading PDF...")
text = read_pdf_text2(pdf_path)

📄 Reading PDF...
Reading page 1/801
Reading page 2/801
Reading page 3/801
Reading page 4/801
Reading page 5/801
Reading page 6/801
Reading page 7/801
Reading page 8/801
Reading page 9/801
Reading page 10/801
Reading page 11/801
Reading page 12/801
Reading page 13/801
Reading page 14/801
Reading page 15/801
Reading page 16/801
Reading page 17/801
Reading page 18/801
Reading page 19/801
Reading page 20/801
Reading page 21/801
Reading page 22/801
Reading page 23/801
Reading page 24/801
Reading page 25/801
Reading page 26/801
Reading page 27/801
Reading page 28/801
Reading page 29/801
Reading page 30/801
Reading page 31/801
Reading page 32/801
Reading page 33/801
Reading page 34/801
Reading page 35/801
Reading page 36/801
Reading page 37/801
Reading page 38/801
Reading page 39/801
Reading page 40/801
Reading page 41/801
Reading page 42/801
Reading page 43/801
Reading page 44/801
Reading page 45/801
Reading page 46/801
Reading page 47/801
Reading page 48/801
Reading page 49/801
Reading page

In [14]:
# =========================================
# SAVE TEXT FILE
# =========================================

txt_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text saved to:", txt_path)

✅ Text saved to: D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt


In [15]:
# READ TEXT FILE
# =========================================

txt_path = r"D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt"
Text = ""

with open(txt_path, "r", encoding="utf-8") as f:
    Text = f.read()

print("✅ Text loaded from:", txt_path)

✅ Text loaded from: D:\GENERATIVE & AGENTIC AI Professional\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\M7 RAG_Retrieval Augmented Generation\RAG - Book project\Deep_Learning_Book_2.txt


In [16]:
text[:500]

'[PAGE 1]\n\n[PAGE 2]\nDeep Learning\nIan Goodfellow\nYoshua Bengio\nAaron Courville\n\n[PAGE 3]\nContents\nWebsite\nvii\nAcknowledgments\nviii\nNotation\nxi\n1\nIntroduction\n1\n1.1\nWho Should Read This Book? . . . . . . . . . . . . . . . . . . . .\n8\n1.2\nHistorical Trends in Deep Learning . . . . . . . . . . . . . . . . .\n11\nI\nApplied Math and Machine Learning Basics\n29\n2\nLinear Algebra\n31\n2.1\nScalars, Vectors, Matrices and Tensors . . . . . . . . . . . . . . .\n31\n2.2\nMultiplying Matrices and Vectors . . . . . . .'

In [17]:
print("✂️ Splitting sections...")
sections = split_sections(text)
len(sections)

✂️ Splitting sections...


523

In [28]:
sections

[{'section': 'general',
  'content': '[PAGE 1]\n\n[PAGE 2]\nDeep Learning\nIan Goodfellow\nYoshua Bengio\nAaron Courville\n\n[PAGE 3]\nContents\nWebsite\nvii\nAcknowledgments\nviii\nNotation\nxi\n1\nIntroduction\n1\n1.1\nWho Should Read This Book? . . . . . . . . . . . . . . . . . . . .\n8\n1.2\nHistorical Trends in Deep Learning . . . . . . . . . . . . . . . . .\n11\nI\nApplied Math and Machine Learning Basics\n29\n2\nLinear Algebra\n31\n2.1\nScalars, Vectors, Matrices and Tensors . . . . . . . . . . . . . . .\n31\n2.2\nMultiplying Matrices and Vectors . . . . . . . . . . . . . . . . . .\n34\n2.3\nIdentity and Inverse Matrices\n. . . . . . . . . . . . . . . . . . . .\n36\n2.4\nLinear Dependence and Span\n. . . . . . . . . . . . . . . . . . . .\n37\n2.5\nNorms . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .\n39\n2.6\nSpecial Kinds of Matrices and Vectors\n. . . . . . . . . . . . . . .\n40\n2.7\nEigendecomposition . . . . . . . . . . . . . . . . . . . . . . . . . .\n42

-- chapters (col) 
-- subsection in chapter (col)

In [ ]:
# =========================================
# 4️ CHUNK SECTIONS
# =========================================
# small docx  >>CHUNK_SIZE  150 to 300 
# artc  >>CHUNK_SIZE  300 to 500 
# pdf for books  >>CHUNK_SIZE  500 to 800 



print("✂️ Chunking sections...")

chunks = build_chunks(
    sections,
    EMBED_MODEL_NAME,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

print("✅ Total chunks:", len(chunks))

✂️ Chunking sections...


Token indices sequence length is longer than the specified maximum sequence length for this model (904 > 512). Running this sequence through the model will result in indexing errors


✅ Total chunks: 3480


In [19]:
# =========================================
# 5️ CREATE EMBEDDINGS
# =========================================

texts = [c["content"] for c in chunks]

print("🧠 Creating embeddings...")
vectors = embed_chunks(texts, embedder)
print("✅ Embeddings created. Sample vector:", vectors[0][:5])

🧠 Creating embeddings...
✅ Embeddings created. Sample vector: [-0.0333883   0.05212121  0.02530358 -0.01006451  0.00404775]


In [20]:
vectors[:2]

array([[-3.33882980e-02,  5.21212108e-02,  2.53035817e-02,
        -1.00645088e-02,  4.04774537e-03,  1.18032442e-02,
         1.49927139e-01, -2.76450329e-02, -1.06364854e-01,
        -7.57755563e-02,  1.73131172e-02,  1.36590973e-01,
         3.77818011e-02, -5.75917289e-02, -7.34874755e-02,
        -3.27040777e-02, -4.91300598e-02, -1.36420606e-02,
        -1.45788621e-02,  7.56727457e-02,  8.39084312e-02,
         7.45033994e-02,  3.36248092e-02,  5.69272004e-02,
         5.71596948e-03,  1.02033541e-02, -3.88757326e-02,
         2.08761008e-03,  1.11626964e-02, -9.10071358e-02,
         3.83642763e-02,  1.45242870e-01, -1.29964091e-02,
         3.53094898e-02,  7.38032609e-02, -7.65414461e-02,
         9.97037292e-02,  3.68791930e-02,  9.97150782e-03,
        -8.22226168e-04,  4.82288711e-02, -3.53691503e-02,
         1.71955884e-03,  3.28749195e-02, -2.34950073e-02,
        -2.19025332e-02, -4.98572625e-02,  8.69028550e-03,
        -4.89821658e-02, -1.53195625e-02, -6.27226755e-0

In [21]:
# ========================================
# 6️ STORE IN DATABASE
# =========================================

print("💾 Storing vectors in PostgreSQL...")

upsert_chunks(
    PG_CONN_STR,
    doc_name,
    chunks,
    vectors
)

💾 Storing vectors in PostgreSQL...


In [26]:
count_chunks(PG_CONN_STR)
preview_chunks(PG_CONN_STR)

📦 Total stored chunks: 3480

🔎 Stored Chunks Preview

DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 1 ]
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 2 ] deep learning ian goodfellow yoshua bengio aaron courville
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
[ page 3 ] contents website vii acknowledgments viii notation xi 1 introduction 1 1. 1 who should read this book?.................... 8 1. 2 historical trends in deep learning................. 11 i ap
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
.................... 36 2. 4 linear dependence and span.................... 37 2. 5 norms................................. 39 2. 6 special kinds of matrices and vectors............... 40 2. 7 eigendec
----------------------------------------
DOC: DeepLearning-IanGoodfellow_RAG
SECTION: general
singular value decomposition......

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=" ", # https://huggingface.co/settings/tokens
)
# ===============================
# Function: Generate Text
# =============================== 
# https://huggingface.co/Qwen/Qwen3-Coder-Next 

def generate_text(context, question,
                  model="Qwen/Qwen3-Coder-Next:novita",
                  temperature=0.2,
                  max_tokens=1024):

    prompt = f"""
    You are an AI assistant that answers questions strictly using the provided CONTEXT.

    Your task is to answer the QUESTION using ONLY the information that appears in the CONTEXT.

    STRICT RULES:
    - Use ONLY the information inside the CONTEXT.
    - Do NOT use prior knowledge or external information.
    - Do NOT guess or infer information that is not explicitly stated.
    - If the answer is not clearly present in the CONTEXT, respond exactly with:
    "The answer is not found in the provided context."
    - Do NOT add explanations that are not supported by the CONTEXT.
    - Prefer quoting or closely paraphrasing the CONTEXT.

    Answer style:
    - Keep the answer clear and concise.
    - Use bullet points when listing multiple facts.

    Output format:

    Answer:
    <answer based strictly on the context>

    Source:
    <section name if available>

    --------------------------------

    CONTEXT:
    {context}

    QUESTION:
    {question}
    """

    try:

        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )

        return completion.choices[0].message.content.strip()

    except Exception as e:

        return f"Error generating response: {str(e)}"

In [ ]:
from IPython.display import display, HTML
import numpy as np


def ask_rag_question(
    question,
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    TOP_K=5
):

    # =========================================
    # 1️⃣ EMBEDDING
    # =========================================
    qvec = embedder.encode(
        [question],
        normalize_embeddings=True
    )[0].astype(np.float32)


    # =========================================
    # 2️⃣ VECTOR SEARCH
    # =========================================
    results = retrieve_topk(
        PG_CONN_STR,
        doc_name,
        qvec,
        TOP_K
    )


    # =========================================
    # 3️⃣ RE-RANK (مستقبلا)
    # =========================================
    reranked = results


    # =========================================
    # 4️⃣ BUILD CONTEXT
    # =========================================
    context = "\n\n".join([
        f"SECTION: {r[0]}\n{r[1]}"
        for r in reranked[:3]
    ])


    # =========================================
    # 5️⃣ GENERATE ANSWER
    # =========================================
    answer = generate_text(context, question)


    # =========================================
    # 6️⃣ FORMAT ANSWER
    # =========================================
    formatted_answer = answer.replace("- ", "• ").replace("\n", "<br>")


    # =========================================
    # 7️⃣ BUILD HTML
    # =========================================
    html = f"""
    <div style="border:2px solid #1976D2; padding:20px; border-radius:10px; font-family:Arial;">

    <h2>🔎 RAG Query Result</h2>

    <p><b>Question:</b> {question}</p>

    <hr>

    <h3>🤖 Generated Answer</h3>

    <div style="
        background:#e3f2fd;
        padding:16px;
        border-radius:10px;
        margin-bottom:20px;
        border-left:6px solid #1976D2;
        font-size:15px;
        line-height:1.6;
        box-shadow:0 2px 6px rgba(0,0,0,0.08);
    ">

    <div style="font-weight:600; margin-bottom:8px;">
    AI Response
    </div>

    <div>
    {formatted_answer}
    </div>

    </div>

    <hr>

    <h3>🏆 Top Retrieved Chunks</h3>
    """


    for r in reranked[:3]:

        section, content, sim, score = r

        html += f"""
        <div style="border:1px solid #ddd; padding:12px; margin-bottom:12px; border-radius:6px;">

        <p><b>Section:</b> {section}</p>
        <p><b>Vector Similarity:</b> {round(sim,3)}</p>
        <p><b>Final Score:</b> {round(score,3)}</p>

        <div style="background:#f5f5f5; padding:10px; border-radius:5px; font-size:14px;">
        {content[:400]}...
        </div>

        </div>
        """

    html += "</div>"

    display(HTML(html))

    return reranked, answer

In [33]:
reranked, answer = ask_rag_question(
    "What are the main parts of the Deep Learning book?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [34]:
reranked, answer = ask_rag_question(
    "What chapter discusses autoencoders?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [40]:
reranked, answer = ask_rag_question(
    "What is backpropagation?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [39]:
reranked[0]

('introduction',
 'nearly as [UNK] to obtain a representation as to solve the original problem, representation learning does not, at ﬁrst glance, seem to help us. deep learning solves this central problem in representation learning by intro - ducing representations that are expressed in terms of other, simpler representations. deep learning allows the computer to build complex concepts out of simpler con - cepts. figure shows how a deep learning system can represent the concept of 1. 2 an image of a person by combining simpler concepts, such as corners and contours, which are in turn deﬁned in terms of edges. the quintessential example of a deep learning model is the feedforward deep network or multilayer perceptron ( mlp ). a multilayer perceptron is just a mathematical function mapping some set of input values to output values. the function is formed by composing many simpler functions.',
 0.6906208005732539,
 0.6906208005732539)

In [41]:
reranked, answer = ask_rag_question(
    "Explain gradient descent.",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)

In [42]:
reranked, answer = ask_rag_question(
    "Explain stochastic gradient descent.",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)